In [1]:
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.by import By
import pandas as pd
import time
import os

# Setup Selenium
driver = webdriver.Chrome()  # Ensure chromedriver is installed and in PATH

url = 'https://tendersodisha.gov.in/nicgep/app?page=WebTenderStatusLists&service=page'
driver.get(url)

# Select Organisation
organisation_name = "WATCO Bhubaneswar"
select = Select(driver.find_element(By.ID, 'OrganName'))
select.select_by_visible_text(organisation_name)

# CAPTCHA Handling
# Download/show CAPTCHA image to user
captcha_img = driver.find_element(By.ID, 'captchaImage').screenshot_as_png
with open('captcha.png', 'wb') as f:
    f.write(captcha_img)
os.startfile('captcha.png')  # Opens the image for user

captcha_value = input("Enter CAPTCHA from image: ")
captcha_input = driver.find_element(By.ID, 'captchaText')
captcha_input.send_keys(captcha_value)

# Click Search
search_btn = driver.find_element(By.ID, 'Search')
search_btn.click()
time.sleep(2)

# Scrape Function
def scrape_page():
    rows = driver.find_elements(By.CSS_SELECTOR, '#tabList tr:not(.list_header)')
    data = []
    for row in rows:
        tds = row.find_elements(By.TAG_NAME, 'td')
        if len(tds) == 6:
            data.append([
                tds[0].text,   # S.No
                tds[1].text,   # Tender ID
                tds[2].text,   # Title and Ref.No.
                tds[3].text,   # Organisation Chain
                tds[4].text,   # Tender Stage
                tds[5].text    # Status (may be empty)
            ])
    return data

# Scrape All Pages
all_data = []
while True:
    all_data.extend(scrape_page())
    # Try to click Next
    try:
        next_btn = driver.find_element(By.LINK_TEXT, 'Next >')
        next_btn.click()
        time.sleep(2)
    except:
        break  # No more pages

# Save to Excel
df = pd.DataFrame(all_data, columns=[
    'S.No', 'Tender ID', 'Title and Ref.No.', 'Organisation Chain', 'Tender Stage', 'Status'
])
df.to_excel('tenders_odisha.xlsx', index=False)
print("Data saved to tenders_odisha.xlsx")

driver.quit()


Enter CAPTCHA from image:  Ydns2e


Data saved to tenders_odisha.xlsx
